# R14: Supervisor Multi-Agent — ORD-50003

Sabse mushkil sawaal — order facts + sahi SOP + decision, teeno chahiye. **Supervisor** kaam specialist agents me baantta hai, aur **plain code** (verifier) final decision release hone se pehle check karta hai.

## Question
"For ORD-50003: was the order late under the right SOP, and what compensation does the customer get?"

## Architecture
1. **Supervisor** (gpt-oss-120b) — route plan: **exactly `data → policy → adjudication → verifier`** isi order me
2. **Specialists**:
   * `data` → UC tool `hop.genai.lookup_order('ORD-50003')` — order facts
   * `policy` → UC tool `hop.genai.search_sop` (2 queries) — `cited_chunk_ids` me 04_Standard SOP ka chunk
   * `adjudication` → LLM, facts + SOP chunks se compensation decision
3. **Verifier** — plain code (no LLM): deterministic rules se expected decision; `agrees` true jab adjudication match kare
4. **Verdict + final answer** — `verdict` = final decision (fault, delivered, is_late, compensation_type, amount, reason_code); agrees hone par hi release

## Attempts
`agent/traces/r14_attempt_<n>.json` — har attempt ka trace (n=1,2,3, saare rakhe jaate hain, **last graded**):
* Attempt 1: decision model llama-3-3-70b (cheap pehle try), prompt me sirf chunks
* Attempt 2: gpt-oss-120b, chunks only
* Attempt 3: gpt-oss-120b + explicit rule checklist (SOP rules prompt me)

**Trace keys (exactly 8):** `question`, `attempt`, `trace_id`, `route` (exactly `["data", "policy", "adjudication", "verifier"]`), `specialists` (data/policy/adjudication), `verifier` (`agrees` + `decision`), `verdict` (6-field final decision), `final_answer`.

In [0]:
# R14 (grader format): supervisor multi-agent for ORD-50003 -> agent/traces/r14_attempt_1/2/3.json
# Route EXACTLY: data -> policy -> adjudication -> verifier. Verdict = final decision (6 fields).
# Verifier = {agrees, decision} — agrees true matlab verdict verifier ke decision se match karta hai.
import json
import uuid

dbutils.widgets.dropdown("catalog", "hop", ["hop"])
catalog = dbutils.widgets.get("catalog")
SUPERVISOR_MODEL = "system.ai.gpt-oss-120b"
QUESTION = "For ORD-50003: was the order late under the right SOP, and what compensation does the customer get?"
ORDER_ID = "ORD-50003"
TRACES_DIR = "/Workspace/HungryHop/Hungry_Hop/Hungry_hop_food_deilvery/src/Gen Ai/agent/traces"
TRACE_KEYS = {"question", "attempt", "trace_id", "route", "specialists", "verifier", "verdict", "final_answer"}
EXPECTED_ROUTE = ["data", "policy", "adjudication", "verifier"]
VERDICT_KEYS = {"fault", "delivered", "is_late", "compensation_type", "amount", "reason_code"}

def _lit(s):
    return "'" + str(s).replace("'", "''") + "'"

def _ai_json(prompt, schema, model):
    sql = ("SELECT ai_query(" + _lit(model) + ", " + _lit(prompt)
           + ", responseFormat => " + _lit(schema) + ") AS r")
    return json.loads(spark.sql(sql).first()["r"])

ROUTE_SCHEMA = json.dumps({
    "type": "json_schema", "json_schema": {"name": "route", "strict": True, "schema": {
        "type": "object",
        "properties": {
            "route": {"type": "array", "items": {"type": "string",
                "enum": ["data", "policy", "adjudication", "verifier"]}}},
        "required": ["route"], "additionalProperties": False}}})

DECISION_SCHEMA = json.dumps({
    "type": "json_schema", "json_schema": {"name": "decision", "strict": True, "schema": {
        "type": "object",
        "properties": {
            "fault": {"type": "string", "enum": ["restaurant", "rider", "customer", "unknown"]},
            "delivered": {"type": "boolean"},
            "is_late": {"type": "boolean"},
            "compensation_type": {"type": "string", "enum": ["NONE", "COUPON", "REFUND"]},
            "amount": {"type": "number"},
            "reason_code": {"type": "string", "enum": ["CUSTOMER_FAULT", "NOT_DELIVERED", "LATE_DELIVERY", "ON_TIME"]},
            "reason": {"type": "string"}},
        "required": ["fault", "delivered", "is_late", "compensation_type", "amount", "reason_code", "reason"],
        "additionalProperties": False}}})

RULES_TEXT = """Compensation rules, first match wins (refund and coupon never combine):
a. Customer fault (CUSTOMER_UNAVAILABLE): NONE, 0, CUSTOMER_FAULT.
b. Otherwise not delivered: REFUND of full order value, NOT_DELIVERED.
c. Otherwise late (delivered AND minutes past promise > class threshold: >5 instant, >20 standard):
   COUPON, LATE_DELIVERY. Amount = base coupon (Instant Rs 40, Standard Rs 60) x tier multiplier
   (free x1, hop_plus x1.5, hop_pro x2) when restaurant or rider at fault; base alone when fault unknown.
d. Otherwise: NONE, 0, ON_TIME.
Fault map: KITCHEN_DELAY/WRONG_ITEM->restaurant, RIDER_DELAY/DAMAGED_IN_TRANSIT->rider,
CUSTOMER_UNAVAILABLE->customer, OTHER/missing->unknown."""

# ---------- specialist agents (route steps) ----------
def specialist_data():
    sql = "SELECT * FROM " + catalog + ".genai.lookup_order(" + _lit(ORDER_ID) + ")"
    return spark.sql(sql).collect()[0].asDict()

def specialist_policy():
    queries = [
        "how many minutes past the promise does a standard order count as late",
        "standard order counts as late coupon amount membership tier multipliers",
    ]
    chunks, seen = [], set()
    for q in queries:
        sql = "SELECT * FROM " + catalog + ".genai.search_sop(" + _lit(q) + ")"
        for r in spark.sql(sql).collect():
            d = r.asDict()
            if d["chunk_id"] not in seen:
                seen.add(d["chunk_id"])
                chunks.append(d)
    return {
        "queries": queries,
        "cited_chunk_ids": [c["chunk_id"] for c in chunks],  # bare chunk ids, best first
        "chunks": [{"chunk_id": c["chunk_id"], "doc_name": c["doc_name"],
                    "heading": c["heading"], "text": c["text"][:200]} for c in chunks],
    }

def specialist_adjudication(facts, chunks, model, with_rules):
    parts = []
    for c in chunks:
        parts.append("[" + c["doc_name"] + " :: " + c["heading"] + "]\n" + c["text"])
    chunk_texts = "\n\n".join(parts)
    prompt = ("You are the compensation decision specialist for HungryHop.\n\n"
              "Order facts: " + json.dumps(facts) + "\n\nRelevant SOP chunks:\n" + chunk_texts + "\n\n")
    if with_rules:
        prompt += "Apply these rules:\n" + RULES_TEXT + "\n\n"
    prompt += "Decide the compensation for this order and reply with the JSON decision."
    return _ai_json(prompt, DECISION_SCHEMA, model)

# ---------- plain-code verifier (no LLM, final step of the route) ----------
def plain_verifier(facts):
    fault_map = {"KITCHEN_DELAY": "restaurant", "WRONG_ITEM": "restaurant",
                 "RIDER_DELAY": "rider", "DAMAGED_IN_TRANSIT": "rider",
                 "CUSTOMER_UNAVAILABLE": "customer"}
    fault = fault_map.get(facts["issue_code"], "unknown")
    mpp = facts["minutes_past_promise"]
    threshold = 5 if facts["delivery_class"] == "instant" else 20
    delivered = facts["status"] == "delivered"
    is_late = delivered and mpp is not None and mpp > threshold
    if fault == "customer":
        exp = {"fault": fault, "delivered": delivered, "is_late": is_late,
               "compensation_type": "NONE", "amount": 0.0, "reason_code": "CUSTOMER_FAULT"}
    elif not delivered:
        exp = {"fault": fault, "delivered": False, "is_late": False,
               "compensation_type": "REFUND", "amount": float(facts["order_value"]), "reason_code": "NOT_DELIVERED"}
    elif is_late:
        base = 40.0 if facts["delivery_class"] == "instant" else 60.0
        mult = {"free": 1.0, "hop_plus": 1.5, "hop_pro": 2.0}[facts["customer_tier"]]
        amount = base * mult if fault in ("restaurant", "rider") else base
        exp = {"fault": fault, "delivered": True, "is_late": True,
               "compensation_type": "COUPON", "amount": amount, "reason_code": "LATE_DELIVERY"}
    else:
        exp = {"fault": fault, "delivered": True, "is_late": False,
               "compensation_type": "NONE", "amount": 0.0, "reason_code": "ON_TIME"}
    return exp, threshold

def compare(decision, expected):
    diffs = [k for k in ("fault", "delivered", "is_late", "compensation_type", "reason_code")
             if decision.get(k) != expected[k]]
    if abs(float(decision.get("amount", -1)) - expected["amount"]) > 0.01:
        diffs.append("amount")
    return diffs

# ---------- one attempt ----------
def run_attempt(n, decision_model, with_rules):
    route_prompt = ("You are the supervisor for the HungryHop support desk. Question: " + QUESTION
                    + "\n\nSplit the work into exactly four steps, in this order: "
                      "1. data (fetch the order's delivery facts via lookup_order), "
                      "2. policy (find the right SOP rules via search_sop), "
                      "3. adjudication (combine the facts and SOPs into the compensation decision), "
                      "4. verifier (plain code checks the decision before it is given). "
                      "Reply with the route as the four step names, in order.")
    route = _ai_json(route_prompt, ROUTE_SCHEMA, SUPERVISOR_MODEL)["route"]
    facts = specialist_data()
    policy = specialist_policy()
    decision = specialist_adjudication(facts, policy["chunks"], decision_model, with_rules)
    expected, threshold = plain_verifier(facts)
    diffs = compare(decision, expected)
    agrees = not diffs
    verifier = {"agrees": agrees, "decision": expected, "differences": diffs}
    if agrees:
        verdict = expected  # released decision — plain code ne approve kiya
    else:  # rejected — verdict 6 fields me specialist ka (unreleased) decision
        verdict = {k: decision[k] for k in ("fault", "delivered", "is_late", "compensation_type", "amount", "reason_code")}
    if agrees:
        cls_name = facts["delivery_class"].capitalize()
        mpp = facts["minutes_past_promise"]
        promised = facts["promised_minutes"]
        tier = facts["customer_tier"]
        final_answer = ("Yes — " + ORDER_ID + " was late under the " + cls_name + " SOP: it was delivered "
                        + format(mpp, ".1f") + " minutes past its " + format(promised, ".0f")
                        + "-minute promise, more than the " + str(threshold) + "-minute threshold. The "
                        + expected["fault"] + " was at fault (" + facts["issue_code"] + "), so the " + tier
                        + " tier applies: the customer gets a " + expected["compensation_type"] + " of Rs "
                        + format(expected["amount"], ".0f") + " (" + expected["reason_code"] + ").")
    else:
        final_answer = ("Attempt " + str(n) + ": the plain-code verifier rejected the adjudication decision "
                        "(mismatched: " + ", ".join(diffs) + "); the supervisor did not release an "
                        "answer and will retry.")
    trace = {
        "question": QUESTION,
        "attempt": n,
        "trace_id": "r14-attempt" + str(n) + "-" + uuid.uuid4().hex[:12],
        "route": route,
        "specialists": {
            "data": facts,
            "policy": policy,
            "adjudication": {"model": decision_model, "rules_in_prompt": with_rules, "decision": decision},
        },
        "verifier": verifier,
        "verdict": verdict,
        "final_answer": final_answer,
    }
    assert set(trace) == TRACE_KEYS
    path = TRACES_DIR + "/r14_attempt_" + str(n) + ".json"
    with open(path, "w") as f:
        json.dump(trace, f, indent=2)
    with open(path) as f:  # read-back verify
        assert json.load(f) == trace
    dm = decision_model.split(".")[-1]
    status = "VERIFIED" if agrees else "REJECTED (" + ", ".join(diffs) + ")"
    print("attempt " + str(n) + " [" + dm + ", rules=" + ("yes" if with_rules else "no") + "]: "
          + status + "  -> " + path)
    return trace, agrees

# ---------- 3 attempts (saare rakhe jaate hain, last graded) ----------
traces = {}
for n, model, rules in [(1, "system.ai.meta-llama-3-3-70b-instruct", False),
                         (2, "system.ai.gpt-oss-120b", False),
                         (3, "system.ai.gpt-oss-120b", True)]:
    traces[n], ok = run_attempt(n, model, rules)

# ---------- final checks (last attempt = graded) ----------
last = traces[3]
assert last["route"] == EXPECTED_ROUTE, "route exactly data, policy, adjudication, verifier order me hona chahiye"
assert last["verifier"]["agrees"] is True, "verifier.agrees true hona chahiye"
assert last["verdict"] == last["verifier"]["decision"], "verdict verifier ke decision ke barabar hona chahiye"
assert set(last["verdict"]) == VERDICT_KEYS, "verdict me exactly 6 decision fields"
assert last["verdict"] == {"fault": "rider", "delivered": True, "is_late": True,
    "compensation_type": "COUPON", "amount": 120.0, "reason_code": "LATE_DELIVERY"}, "ORD-50003 ka decision galat"
pol = last["specialists"]["policy"]
assert isinstance(pol["cited_chunk_ids"], list) and all(isinstance(x, str) for x in pol["cited_chunk_ids"])
std_ids = [c["chunk_id"] for c in pol["chunks"] if c["doc_name"] == "04_Late_Delivery_Compensation_Standard.md"]
assert std_ids and any(i in pol["cited_chunk_ids"] for i in std_ids), "cited_chunk_ids me 04_Standard SOP ka chunk hona chahiye"
assert "late" in last["final_answer"].lower() and "120" in last["final_answer"]
for n in (1, 2, 3):
    with open(TRACES_DIR + "/r14_attempt_" + str(n) + ".json") as f:
        t = json.load(f)
    assert set(t) == TRACE_KEYS and t["attempt"] == n and isinstance(t["route"], list) \
        and isinstance(t["specialists"], dict) and isinstance(t["verifier"], dict) and isinstance(t["verdict"], dict)
    assert isinstance(t["verifier"].get("agrees"), bool) and set(t["verdict"]) == VERDICT_KEYS
print("\nR14 DONE — 3 attempt traces (grader format) saved; last attempt: route data->policy->adjudication->verifier, "
      "verifier.agrees=True, verdict = COUPON Rs 120 (rider, hop_pro x2, 22 min > 20)")
print("\nfinal_answer (attempt 3): " + last["final_answer"])